# Vigía de aseo — entrenamiento de heces (RF-DETR, v3: heces + basura)
**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo → GPU T4*.
Datos: ScatSpotter (Crall), CC BY 4.0.

Ejecuta las celdas **en orden**.

In [ ]:
!nvidia-smi -L

## 1. Limpiar corridas anteriores
Solo hace falta si ya usaste este cuaderno antes.

In [ ]:
!rm -rf dataset corrida corrida_humo corrida_multi export umbral*.json modelo_vigia.zip colab_paquete_v2.zip entrenamiento

## 2. Traer el paquete desde Google Drive
1. En tu PC sube `colab_paquete_v2.zip` (760 MB) a **Mi unidad** de Google Drive (drive.google.com → Nuevo → Subir archivo). Espera a que Drive diga que terminó.
2. Ejecuta la celda: te pedirá permiso para conectar Drive.

La celda comprueba el tamaño: debe ser **760426006 bytes**. Si no coincide, la subida a Drive no terminó.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/colab_paquete_v2.zip /content/colab_paquete_v2.zip
import os
tam = os.path.getsize('/content/colab_paquete_v2.zip')
print('tamaño:', tam, '(esperado 760426006)')
assert tam == 760426006, 'El zip está incompleto: espera a que termine de subir a Drive y repite esta celda'

## 2b. Descomprimir

In [ ]:
!unzip -q -o colab_paquete_v2.zip
!ls dataset dataset/train | head

## 3. Instalar
Si pide reiniciar el entorno, reinícialo y vuelve a ejecutar desde el paso 2 (sin borrar el zip).

In [ ]:
!pip install "rfdetr[train,loggers,onnx]==1.11.2" onnxruntime pillow 2>&1 | tail -25
!python -c "import rfdetr; print('rfdetr OK', rfdetr.__file__)"

## 4. Entrenar (modelo `small`, 512 px)
Déjalo hasta que termine solo (parada anticipada). Debe terminar con `Listo. Mejor checkpoint`.

In [ ]:
!python entrenamiento/entrenar.py --dataset dataset --tamano small --epocas 50 --salida corrida_multi

## 5. Evaluar y elegir umbral
El umbral se elige en *valid*; *test* se mira una sola vez al final. Pega ambas tablas en el chat.

In [ ]:
!python entrenamiento/evaluar.py --checkpoint corrida_multi/checkpoint_best_total.pth --dataset dataset --parte valid --salida umbral.json
!python entrenamiento/evaluar.py --checkpoint corrida_multi/checkpoint_best_total.pth --dataset dataset --parte test --salida umbral_test.json

## 6. Exportar a ONNX

In [ ]:
!python entrenamiento/exportar.py --checkpoint corrida_multi/checkpoint_best_total.pth --umbral umbral.json --salida export
!cd export && zip -r ../modelo_vigia.zip . && cd ..
!ls -lh modelo_vigia.zip

## 7. Descargar el modelo
(celda de Python aparte)

In [ ]:
from google.colab import files
files.download('modelo_vigia.zip')